# **GPT-1 실습**
: Implementation of GPT-1 model, including pre-training & fine-tuning process. Pre-trained on WikiText2, fine-tuned on IMDB Dataset.
## Improving Language Understanding by Generative Pre-Training

⏩ 논문링크: https://www.mikecaptain.com/resources/pdf/GPT-1.pdf


⏩ 깃허브: https://github.com/tony3ynot/GPT-1/blob/main/GPT_1.ipynb 코드를 참고했습니다


📑 추가로 참고해볼만한 깃허브: https://github.com/lyeoni/gpt-pytorch



In [1]:
# 필요한 라이브러리 임포트
import torch                          # PyTorch 메인 모듈
import torch.nn as nn                 # 신경망 레이어 모듈
from einops import rearrange          # 텐서 차원 재배열을 위한 라이브러리 (멀티헤드 분리/병합 시 사용)


In [2]:
# Google Colab에서 Google Drive 마운트 (모델 저장/불러오기용)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#1. Model Architecture




## 1-1. Transformer Decoder

In [3]:
### Multi-Head Attention (멀티헤드 어텐션)
class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        """
        d_model: 모델의 임베딩 차원 (예: 768)
        n_heads: 어텐션 헤드 개수 (예: 12)
        """
        super().__init__()
        self.n_heads = n_heads

        # Q, K, V를 만들기 위한 선형 투영 레이어
        self.fc_q = nn.Linear(d_model, d_model) # Query (질의) 투영
        self.fc_k = nn.Linear(d_model, d_model) # Key (키) 투영
        self.fc_v = nn.Linear(d_model, d_model) # Value (값) 투영

        self.fc = nn.Linear(d_model, d_model)   # 멀티헤드 결과를 합친 후 통과시키는 출력 선형 레이어

        # 스케일링 인자: sqrt(d_k) -- 어텐션 점수가 너무 커지는 것을 방지
        self.scale = torch.sqrt(torch.tensor(d_model/n_heads))

    def forward(self, Q, K, V, mask = None):
        # 1) 입력을 Q, K, V로 선형 투영
        Q = self.fc_q(Q)
        K = self.fc_k(K)
        V = self.fc_v(V)

        ## B = 배치 크기 / L = 시퀀스 길이 / H = 헤드 개수 / D = 헤드별 차원
        # 멀티헤드 처리를 위해 차원을 재배열: (B, L, d_model) -> (B, H, L, D)
        Q = rearrange(Q, 'B L (H D) -> B H L D', H = self.n_heads)
        K = rearrange(K, 'B L (H D) -> B H L D', H = self.n_heads)
        V = rearrange(V, 'B L (H D) -> B H L D', H = self.n_heads)

        ## Self-Attention (자기 어텐션) 계산 단계
        # 1. MatMul: Q와 K^T의 행렬곱으로 어텐션 점수(유사도) 계산
        attention_score = Q @ K.transpose(-2, -1)

        # 2. Scale: sqrt(d_k)로 나누어 스케일링 (수치 안정성 확보)
        attention_score = attention_score / self.scale

        # 3. Masking: 패딩 토큰 또는 미래 토큰 위치를 매우 작은 값(-1e9)으로 설정
        #             -> softmax를 거치면 사실상 0이 되어 무시됨
        if mask is not None:
            mask = mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)  # 헤드 차원으로 확장
            attention_score.masked_fill_(mask, -1e9)

        # 4. SoftMax: 마지막 차원(키 위치)에 대해 확률 분포로 변환
        attention_weights = torch.softmax(attention_score, dim=-1)

        # 5. MatMul: 어텐션 가중치와 V의 행렬곱 -> 가중합된 표현 산출
        attention = attention_weights @ V

        ## Concat & Linear (헤드들 결합 후 선형 변환)
        # 헤드를 다시 합침: (B, H, L, D) -> (B, L, H*D=d_model)
        x = rearrange(attention, 'B H L D -> B L (H D)')
        output = self.fc(x)  # 마지막 선형 변환

        return output


### Feed Forward Network (위치별 피드포워드 네트워크)
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        d_model: 입력/출력 차원
        d_ff: 은닉 차원 (보통 d_model의 4배, 예: 3072)
        """
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)   # 차원 확장
        self.linear2 = nn.Linear(d_ff, d_model)   # 차원 축소 (원래대로)
        self.gelu = nn.GELU()                     # GPT-1에서 사용하는 활성화 함수 (ReLU 대신 GELU)

        # Xavier 정규 초기화로 학습 안정화
        nn.init.xavier_normal_(self.linear1.weight)
        nn.init.xavier_normal_(self.linear2.weight)

    def forward(self, x):
        x = self.gelu(self.linear1(x))   # 확장 + GELU 비선형
        output = self.linear2(x)         # 다시 d_model로 축소

        return output


### Decoder Layer (디코더 한 층 = MHA + FFN, 각각 잔차연결과 LayerNorm 포함)
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, resid_drop):
        super().__init__()

        # 첫 번째 서브레이어: Masked Multi-Head Attention
        self.mha = MHA(d_model, n_heads)
        self.dropout1 = nn.Dropout(resid_drop)
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-5)

        # 두 번째 서브레이어: Feed Forward Network
        self.ffn = FFN(d_model, d_ff)
        self.dropout2 = nn.Dropout(resid_drop)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, x, attn_mask):
        # 1) Masked-MHA 서브레이어 (잔차 연결 + LayerNorm)
        # GPT-1은 self-attention이므로 Q, K, V에 모두 같은 x를 입력
        residual = self.mha(x, x, x, attn_mask)
        residual = self.dropout1(residual)
        x = self.layernorm1(x + residual)   # Post-LN 방식 (원본 Transformer 방식)

        # 2) FFN 서브레이어 (잔차 연결 + LayerNorm)
        residual = self.ffn(x)
        residual = self.dropout2(residual)
        output = self.layernorm2(x + residual)

        return output


### Decoder (전체 디코더: 임베딩 + 위치 임베딩 + N개의 DecoderLayer)
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model, n_layers, n_heads, d_ff, embd_drop, resid_drop, pad_id):
        super().__init__()

        self.pad_id = pad_id  # 패딩 토큰 ID (마스킹용)

        ## Decoder Input (입력 임베딩)
        self.embedding = nn.Embedding(vocab_size, d_model)        # 토큰 임베딩
        self.dropout = nn.Dropout(embd_drop)                      # 임베딩 드롭아웃
        self.pos_embedding = nn.Embedding(seq_len+1, d_model)     # 학습 가능한 위치 임베딩 (sinusoidal 아님)
                                                                  # +1은 패딩 위치(0)를 위한 여유분

        ## Decoder Layers (디코더 레이어 N개를 쌓음)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, resid_drop) for _ in range(n_layers)])

        nn.init.xavier_normal_(self.embedding.weight)

    def forward(self, x):
        ## 위치 인덱스 생성 (패딩 위치는 0번 위치 임베딩 사용)
        positions = torch.arange(x.size(1), device=x.device).repeat(x.size(0), 1) + 1
        position_pad_mask = x.eq(self.pad_id)             # 패딩이 있는 자리 식별
        positions.masked_fill_(position_pad_mask, 0)      # 패딩 위치는 위치 인덱스를 0으로 설정

        # 토큰 임베딩 + 위치 임베딩 (드롭아웃 적용)
        output = self.dropout(self.embedding(x)) + self.pos_embedding(positions)

        ## 어텐션 마스크 생성 (패딩 마스크 + 미래 토큰 마스크 결합)
        pad_mask = self.get_padding_mask(x, x, self.pad_id)
        future_mask = self.get_future_mask(x).to(device=pad_mask.device)
        # 둘 중 하나라도 마스킹 대상이면 True로 -> 자기회귀(autoregressive) 디코딩 보장
        attn_mask = torch.gt((pad_mask.to(dtype=future_mask.dtype) + future_mask), 0)

        # 모든 디코더 레이어를 차례로 통과
        for layer in self.layers:
            output = layer(output, attn_mask)

        return output

    ## padding mask: 패딩 토큰 위치를 마스킹 (어텐션이 패딩에 주목하지 않도록)
    def get_padding_mask(self, q, k, pad_id):
        pad_mask = k.eq(pad_id).unsqueeze(1).repeat(1, q.size(1), 1)

        return pad_mask

    ## future token mask: 미래 토큰 마스킹 (Causal Masking)
    # GPT는 다음 단어 예측이므로, 현재 위치에서 미래 토큰을 보지 못하게 막음
    def get_future_mask(self, q):
        bs, q_len = q.size()
        # 상삼각 행렬(diagonal=1) -> 자기 자신은 보되 미래만 가림
        future_mask = torch.ones(bs, q_len, q_len).triu(diagonal=1)

        return future_mask


## 1-2. GPT-1

In [4]:
### GPT-1 본체 (Transformer Decoder만 래핑)
class GPT(nn.Module):
    def __init__(self,
                 vocab_size,           # 어휘집(vocabulary) 크기
                 seq_len = 512,        # 최대 시퀀스 길이
                 d_model = 768,        # 임베딩/은닉 차원 (GPT-1 원본 설정)
                 n_layers = 12,        # 디코더 레이어 수
                 n_heads = 12,         # 멀티헤드 어텐션 헤드 수
                 d_ff = 3072,          # FFN 은닉 차원 (= 4 * d_model)
                 embd_drop = 0.1,      # 임베딩 드롭아웃 비율
                 resid_drop = 0.1,     # 잔차 연결 드롭아웃 비율
                 pad_id = 0):          # 패딩 토큰 ID
        super().__init__()

        # GPT-1은 Transformer의 디코더만 사용 (인코더 없음)
        self.decoder = TransformerDecoder(vocab_size, seq_len, d_model, n_layers, n_heads,
                                          d_ff, embd_drop, resid_drop, pad_id)

    def forward(self, x):
        outputs = self.decoder(x)   # (B, L, d_model) 형태의 은닉 표현 반환

        return outputs


### Language Model Head (사전학습용: 다음 토큰 예측)
class GPTLMHead(nn.Module):
    def __init__(self, gpt):
        super().__init__()
        vocab_size, d_model = gpt.decoder.embedding.weight.size()

        self.gpt = gpt
        # 은닉 표현을 어휘집 크기로 투영하는 선형 레이어 (bias 없음)
        self.linear = nn.Linear(d_model, vocab_size, bias = False)
        # Weight Tying: 입력 임베딩과 출력 투영의 가중치를 공유 (파라미터 절약 + 성능 향상)
        self.linear.weight = gpt.decoder.embedding.weight

    def forward(self, x):
        x = self.gpt(x)                # GPT 본체 통과: (B, L, d_model)

        lm_logits = self.linear(x)     # 어휘집 크기 로짓: (B, L, vocab_size)

        return lm_logits


### Classification Head (파인튜닝용: 문장 분류)
class GPTClsHead(nn.Module):
    def __init__(self, gpt, n_class, cls_token_id, cls_drop=0.1):
        """
        gpt: 사전학습된 GPT 모델
        n_class: 분류 클래스 개수 (IMDB의 경우 2 -> 긍정/부정)
        cls_token_id: <cls> 토큰의 ID (분류용 표현 추출 위치)
        cls_drop: 분류 헤드의 드롭아웃 비율
        """
        super().__init__()
        vocab_size, d_model = gpt.decoder.embedding.weight.size()
        self.cls_token_id = cls_token_id

        self.gpt = gpt

        # LM 헤드 (보조 손실용) -- 사전학습 헤드 그대로 유지
        self.linear1 = nn.Linear(d_model, vocab_size, bias=False)
        self.linear1.weight = gpt.decoder.embedding.weight   # Weight tying
        # Cls 헤드 (분류용)
        self.linear2 = nn.Linear(d_model, n_class)
        self.dropout = nn.Dropout(cls_drop)

        # 분류 헤드는 GPT 논문 권장대로 정규분포(std=0.02)로 초기화
        nn.init.normal_(self.linear2.weight, std=0.02)
        nn.init.normal_(self.linear2.bias, 0)

    def forward(self, x):
        outputs = self.gpt(x)                              # (B, L, d_model) 은닉 표현

        lm_logits = self.linear1(outputs)                  # 언어모델 로짓 (보조 손실에 사용)

        # <cls> 토큰 위치의 은닉 표현만 추출 -> 분류용 입력
        outputs = outputs[x.eq(self.cls_token_id)]
        cls_logits = self.linear2(self.dropout(outputs))   # 분류 로짓

        return lm_logits, cls_logits


# 2. Training

## 2-1. Pre-training

In [5]:
# 필요한 패키지 설치 (Hugging Face 라이브러리들)
!pip install transformers datasets tokenizers

# 학습/평가에 필요한 추가 모듈 임포트
import torch.nn.functional as F                # 손실 함수, 활성화 함수 등
from torch.utils.data import Dataset, DataLoader  # 데이터셋/데이터로더
from datasets import load_dataset              # Hugging Face 데이터셋 로더
from tokenizers import Tokenizer               # 토크나이저
from tokenizers.models import BPE              # Byte-Pair Encoding 모델
from tokenizers.trainers import BpeTrainer     # BPE 토크나이저 학습기
from tokenizers.pre_tokenizers import Whitespace  # 공백 기반 사전 토큰화
import numpy as np
from tqdm import tqdm                          # 진행 상황 표시

# GPU 사용 가능하면 GPU, 아니면 CPU 사용
device = "cuda" if torch.cuda.is_available() else "cpu"


In [6]:
### WikiText 데이터셋 클래스 (사전학습용)
class WikiTextDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        """
        data: Hugging Face 데이터셋 객체
        tokenizer: 학습된 BPE 토크나이저
        seq_len: 시퀀스 길이 (입력 + 타겟 분리를 위해 실제로는 SEQ_LEN+1 사용)
        """
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        encoded = self.tokenizer.encode(text)
        input_ids = encoded.ids

        # 시퀀스 길이 맞추기: 길면 자르고(truncate), 짧으면 패딩 추가
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))   # 0번이 <pad>

        # 다음 단어 예측을 위한 입력/타겟 분리
        # inputs[t]를 보고 targets[t] = inputs[t+1]을 예측하도록 학습
        inputs = torch.tensor(input_ids[:-1])
        targets = torch.tensor(input_ids[1:])

        return inputs, targets


In [7]:
### 하이퍼파라미터 설정
VOCAB_SIZE = 10000        # 어휘집 크기 (BPE로 학습할 토큰 개수)
SEQ_LEN = 512             # 최대 시퀀스 길이
BATCH_SIZE = 8            # 배치 크기
EPOCHS = 3                # 학습 에폭 수
LEARNING_RATE = 1e-4      # 학습률 (사전학습용)

### 토크나이저 학습 (BPE 기반)
# WikiText-2 데이터셋 로드 (raw 버전 사용)
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
tokenizer = Tokenizer(BPE())                                                  # BPE 모델로 토크나이저 생성
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<cls>"])  # 특수 토큰 정의
tokenizer.pre_tokenizer = Whitespace()                                        # 공백 기준으로 우선 분할

# 학습용 코퍼스 제너레이터 (메모리 효율적으로 데이터 공급)
def get_training_corpus():
    for i in range(0, len(dataset['train'])):
        yield dataset['train'][i]['text']

# 토크나이저 학습 시작
tokenizer.train_from_iterator(get_training_corpus(), trainer)

### 데이터셋 및 DataLoader 설정
# SEQ_LEN+1을 쓰는 이유: input/target 분리 시 한 토큰씩 어긋나기 때문
train_dataset = WikiTextDataset(dataset['train'], tokenizer, SEQ_LEN + 1)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [8]:
### 모델 초기화 (사전학습용 LM Head 부착)
model = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)

# AdamW 옵티마이저 (Adam + 가중치 감쇠 분리)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# Cosine Annealing 스케줄러: 학습률을 코사인 곡선 형태로 점차 감소
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_dataloader) * EPOCHS)


In [ ]:
### 사전학습 (Pre-Training) 루프
for epoch in range(EPOCHS):
    model.train()         # 학습 모드 (드롭아웃 등 활성화)
    total_loss = 0

    # tqdm으로 진행률 시각화
    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        inputs, targets = inputs.to(device), targets.to(device)

        # 순전파: 다음 토큰 예측 로짓 계산
        logits = model(inputs)
        # Cross Entropy 손실 계산 (패딩 토큰 0은 손실 계산에서 제외)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=0)

        # 역전파 및 파라미터 업데이트
        optimizer.zero_grad()                                          # 그래디언트 초기화
        loss.backward()                                                # 그래디언트 계산
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)        # 그래디언트 클리핑 (폭발 방지)
        optimizer.step()                                               # 파라미터 갱신
        scheduler.step()                                               # 학습률 갱신

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})  # 평균 손실 표시

    avg_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("Training completed!")

# 사전학습 완료 후 체크포인트 저장 (모델/옵티마이저/스케줄러 상태 모두 저장)
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'final_loss': avg_loss
}, 'gpt1_pretrained.pt')


Epoch 1/3: 100%|██████████| 4590/4590 [1:02:57<00:00,  1.21it/s, loss=7.13]



Epoch 1 Average Loss: 7.1329


Epoch 2/3:  95%|█████████▌| 4362/4590 [59:51<03:08,  1.21it/s, loss=nan]

## 2-2. Fine-tuning

In [ ]:
### IMDB 데이터셋 클래스 (파인튜닝용 -- 영화 리뷰 감성 분류)
class IMDBDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']        # 리뷰 텍스트
        label = self.data[idx]['label']      # 0(부정) 또는 1(긍정)

        # 시퀀스 맨 앞에 <cls> 토큰을 추가 -- 분류 표현을 추출할 위치
        encoded = self.tokenizer.encode("<cls> " + text)
        input_ids = encoded.ids

        # 시퀀스 길이 맞추기 (자르기 또는 패딩)
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))

        return torch.tensor(input_ids), torch.tensor(label)


In [ ]:
### IMDB 데이터셋 로드 및 DataLoader 설정
from datasets import load_dataset
imdb_dataset = load_dataset('imdb')   # IMDB 영화 리뷰 데이터셋 (긍정/부정 이진 분류)

# 학습용 데이터로더
train_dataset = IMDBDataset(imdb_dataset['train'], tokenizer, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
# 검증용 데이터로더
val_dataset = IMDBDataset(imdb_dataset['test'], tokenizer, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=4)


In [ ]:
# 사전학습된 GPT 모델 불러오기
premodel = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)
premodel.load_state_dict(torch.load('gpt1_pretrained.pt'))

### 파인튜닝 모델 초기화
# 사전학습된 GPT 본체에 분류 헤드를 부착
model = GPTClsHead(
    gpt=premodel.gpt,                                # 사전학습된 GPT 본체
    n_class=2,                                       # 클래스 수: 긍정/부정
    cls_token_id=tokenizer.token_to_id("<cls>"),     # <cls> 토큰 ID 전달
    cls_drop=0.1                                     # 분류 헤드 드롭아웃
).to(device)

# 파인튜닝은 더 작은 학습률 사용 (1e-5) -- 사전학습 가중치 망가지지 않게
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)


In [ ]:
### 파인튜닝 (Fine-tuning) 루프
EPOCHS = 3
auxiliary_ratio = 0.5     # GPT-1 논문의 보조 LM 손실 가중치 (lambda)
best_acc = 0              # 최고 정확도 추적

for epoch in range(EPOCHS):
    ## 학습 단계
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        inputs, labels = inputs.to(device), labels.to(device)

        # 순전파: LM 로짓과 분류 로짓을 동시에 얻음
        lm_logits, cls_logits = model(inputs)
        # LM 로짓은 마지막 토큰을 제외 (다음 토큰 예측이므로 한 칸 어긋남)
        lm_logits = lm_logits[:, :-1].contiguous()

        ## 손실 함수 (GPT-1 논문의 보조 손실 방식)
        # L1: LM 보조 손실 (next token prediction) -- 사전학습 지식 유지
        lm_loss = F.cross_entropy(lm_logits.view(-1, lm_logits.size(-1)),
                                  inputs[:, 1:].contiguous().view(-1), ignore_index=0)
        # L2: 분류 손실 (메인 태스크)
        cls_loss = F.cross_entropy(cls_logits, labels)
        # L3: 최종 손실 = 분류 손실 + λ * LM 보조 손실
        loss = cls_loss + (auxiliary_ratio * lm_loss)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)   # 그래디언트 클리핑
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})

    ## 검증 단계 (정확도 측정)
    model.eval()      # 평가 모드 (드롭아웃 비활성화)
    correct = 0
    total = 0

    with torch.no_grad():    # 그래디언트 계산 비활성화 (메모리/속도 절감)
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, cls_logits = model(inputs)

            # 가장 높은 로짓을 가진 클래스를 예측값으로 선택
            predictions = torch.argmax(cls_logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Epoch {epoch+1} Validation Accuracy: {accuracy:.4f}")

    # 검증 정확도가 갱신되면 최고 모델 저장
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), 'gpt1_imdb_best.pt')

print(f"Fine-tuning completed! Best accuracy: {best_acc:.4f}")


# Test

In [ ]:
# 저장된 최고 모델 불러오기 (테스트용)
model = GPTClsHead(
    GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN),
    n_class=2,
    cls_token_id=tokenizer.token_to_id("<cls>"),
    cls_drop=0.1
)

# 파인튜닝 후 가장 좋았던 가중치 로드
model.load_state_dict(torch.load('gpt1_imdb_best.pt'))


In [ ]:
# 테스트: 임의의 텍스트에 대해 감성 예측
def predict_sentiment(text):
    """입력 텍스트에 대해 긍정/부정을 예측하는 함수"""
    model.eval()       # 평가 모드 전환

    # IMDBDataset과 동일한 방식으로 전처리
    encoded = tokenizer.encode("<cls> " + text)        # <cls> 토큰 추가
    input_ids = encoded.ids

    # 길이 맞추기 (자르기 또는 패딩)
    if len(input_ids) > SEQ_LEN:
        input_ids = input_ids[:SEQ_LEN]
    else:
        input_ids = input_ids + [0] * (SEQ_LEN - len(input_ids))

    # 배치 차원 추가 후 디바이스로 이동
    inputs = torch.tensor([input_ids]).to(device)

    with torch.no_grad():
        _, cls_logits = model(inputs)                   # 분류 로짓 추출
        prediction = torch.argmax(cls_logits, dim=-1)   # 최대 로짓 클래스 선택

    return "Positive" if prediction.item() == 1 else "Negative"

# 사용 예시
test_text = "This movie was really great! I enjoyed every moment of it."
print(f"Text: {test_text}")
print(f"Sentiment: {predict_sentiment(test_text)}")
